# Residual Signal Finder V2 on UCI credit data

This notebook demonstrates `ResidualSignalFinderV2`, the stability-oriented residual
signal finder. It uses the UCI **Default of Credit Card Clients** dataset and mirrors
the original residual finder walkthrough while emphasizing the V2 workflow:

1. Fit a simple GLM-style base model and create out-of-fold base predictions.
2. Define residuals as `actual - base_prediction`.
3. Run optional multivariate screening with permutation importance.
4. Score candidate features with univariate residual lift across repeated splits.
5. Review residual lift, rank stability, Spearman direction, effect-curve stability, and null/shadow baselines.

## Setup

The path setup lets the notebook run from the repository root or from inside
`notebooks/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in [PROJECT_ROOT / 'src', PROJECT_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT

In [ ]:
import warnings

import matplotlib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pe_tools.signal_finder import ResidualSignalFinderV2

warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## Load the UCI credit dataset

The saved CSV has 30,000 rows and 23 numeric modeling features after dropping the
target. V2 can also evaluate categorical candidate features, so this walkthrough adds
a simple `age_band` feature for diagnostics.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'default_of_credit_card_clients.csv'
credit = pd.read_csv(DATA_PATH)
target_col = 'default_next_month'
base_feature_cols = [column for column in credit.columns if column != target_col]

print(f'Rows: {len(credit):,}')
print(f'Base feature count: {len(base_feature_cols)}')
display(credit.head())
display(credit[target_col].value_counts(normalize=True).rename('target_rate'))

## Create a fast analysis sample

The full data remain in `data/`; the notebook uses a stratified sample so V2's
bootstrap diagnostics run quickly. Explicit train/validation/holdout index sets are
created later to demonstrate the `splits=` API for TVH workflows.


In [ ]:
analysis_frame = train_test_split(
    credit,
    train_size=2_500,
    stratify=credit[target_col],
    random_state=42,
)[0].sort_index()

train_valid, holdout_rows = train_test_split(
    analysis_frame,
    test_size=0.15,
    stratify=analysis_frame[target_col],
    random_state=42,
)
train_rows, validation_rows = train_test_split(
    train_valid,
    test_size=0.1765,
    stratify=train_valid[target_col],
    random_state=42,
)

X_base = analysis_frame.loc[:, base_feature_cols].copy()
y_true = analysis_frame[target_col].astype(float).rename(target_col)
split_label = pd.Series('train', index=X_base.index, name='split')
split_label.loc[validation_rows.index] = 'validation'
split_label.loc[holdout_rows.index] = 'holdout'

split_summary = pd.concat(
    [
        split_label.value_counts().rename('rows'),
        y_true.groupby(split_label).mean().rename('event_rate'),
    ],
    axis=1,
)
display(split_summary)


## Build sample weights and candidate features

Weights are optional. Here they balance the minority default class and mildly emphasize
larger credit limits. `age_band` is intentionally not part of the base GLM, so V2 can
show how a new candidate feature is handled.

In [ ]:
default_rate = float(y_true.mean())
class_weight = np.where(
    y_true.eq(1),
    0.5 / default_rate,
    0.5 / (1.0 - default_rate),
)
limit_weight = (X_base['limit_bal'] / X_base['limit_bal'].median()).clip(0.25, 4.0)
sample_weight = pd.Series(
    class_weight * limit_weight.to_numpy(),
    index=X_base.index,
    name='sample_weight',
)

X_candidates = X_base.copy()
X_candidates['age_band'] = pd.cut(
    X_base['age'],
    bins=[20, 30, 40, 50, 60, 80],
    include_lowest=True,
).astype(str)

print(f'Candidate feature count: {X_candidates.shape[1]}')
display(sample_weight.describe())
display(X_candidates[['age', 'age_band', 'limit_bal', 'pay_0']].head())

## Create out-of-fold base predictions

V2 assumes `base_pred` is preferably out-of-fold or out-of-sample. This cell creates
OOF predictions from a simple standardized logistic regression that uses all numeric
base features.

In [ ]:
oof_pred = pd.Series(np.nan, index=X_base.index, name='oof_default_probability')
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

for fold, (train_idx, validation_idx) in enumerate(cv.split(X_base, y_true), start=1):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1_000, solver='lbfgs'),
    )
    model.fit(
        X_base.iloc[train_idx],
        y_true.iloc[train_idx],
        logisticregression__sample_weight=sample_weight.iloc[train_idx],
    )
    oof_pred.iloc[validation_idx] = model.predict_proba(X_base.iloc[validation_idx])[:, 1]
    fold_auc = roc_auc_score(y_true.iloc[validation_idx], oof_pred.iloc[validation_idx])
    print(f'Fold {fold} ROC AUC: {fold_auc:.3f}')

print(f'OOF ROC AUC: {roc_auc_score(y_true, oof_pred):.3f}')
display(pd.DataFrame({'actual': y_true, 'base_pred': oof_pred}).head())

## Fit `ResidualSignalFinderV2`

For notebook speed, this run uses random forests with modest tree counts. The same API
supports `univariate_model_type='xgboost'` and `screening_model_type='xgboost'`.

In [ ]:
finder = ResidualSignalFinderV2(
    screening_enabled=True,
    screening_model_type='random_forest',
    screening_top_k=8,
    screening_cv_folds=3,
    screening_n_repeats=1,
    univariate_model_type='random_forest',
    n_bootstraps=8,
    test_size=0.25,
    n_bins=8,
    n_null_features=4,
    random_state=42,
    use_sample_weight=True,
    model_params={'n_estimators': 30, 'max_depth': 3, 'min_samples_leaf': 25},
)

finder.fit(
    X_candidates,
    y=y_true,
    base_pred=oof_pred,
    sample_weight=sample_weight,
    original_model_features=base_feature_cols,
)

summary = finder.get_summary()
display(pd.Series(finder.residual_summary_, name='value').to_frame())
display(summary.head(10))

## Inspect V2 result tables

V2 stores feature-level summaries, per-bootstrap scores, effect curves, null-feature
scores, and optional screening results directly on the fitted finder.

In [ ]:
print('Screening results')
display(finder.screening_results_.head(10))

print('Bootstrap results')
display(finder.bootstrap_results_.head(10))

print('Effect curves')
display(finder.effect_curves_.head(10))

print('Null results')
display(finder.null_results_.head(10))

print('Warnings')
display(finder.warnings_)

## Feature-level lookup

`get_feature_summary` returns one feature's compact V2 stability row.

In [ ]:
top_feature = str(summary.iloc[0]['feature'])
print(f'Top feature: {top_feature}')
display(finder.get_feature_summary(top_feature).to_frame('value'))

## Plot diagnostics for one feature

The diagnostic figure includes actual/base/corrected prediction by bin, residual mean
curves, residual model effects, bootstrap spaghetti curves, residual distributions,
absolute residuals, and the null comparison.

In [ ]:
figure = finder.plot_feature_diagnostics(top_feature)
display(figure)
plt.close(figure)

## Plot focused V2 diagnostics

These methods return matplotlib `Figure` objects and do not save files unless you do
so explicitly with matplotlib.

In [ ]:
figures = {
    'effect_curve': finder.plot_effect_curve(top_feature),
    'null_comparison': finder.plot_null_comparison(top_feature),
    'rank_stability': finder.plot_rank_stability(n=8),
    'residual_signal_map': finder.plot_residual_signal_map(),
}

for name, figure in figures.items():
    print(f'Displaying {name}')
    display(figure)
    plt.close(figure)

## Plot top features

`plot_top_features` is useful when reviewing several candidates after sorting by
`mean_oof_residual_r2`.

In [ ]:
top_figures = finder.plot_top_features(n=2)
for feature, figure in top_figures.items():
    print(f'Displaying diagnostics for {feature}')
    display(figure)
    plt.close(figure)

## User-supplied split example

This compact run uses the explicit train/validation/holdout labels created earlier.
Rows outside `train` are treated as the validation set for a single diagnostic split.

In [ ]:
compact_features = summary.head(6)['feature'].astype(str).tolist()
tvh_splits = [
    {
        'train': train_rows.index,
        'validation': validation_rows.index,
        'holdout': holdout_rows.index,
    }
]
custom_split_finder = ResidualSignalFinderV2(
    screening_enabled=False,
    univariate_model_type='random_forest',
    n_bootstraps=1,
    n_bins=6,
    n_null_features=3,
    random_state=42,
    use_sample_weight=True,
    model_params={'n_estimators': 25, 'max_depth': 3, 'min_samples_leaf': 20},
)
custom_split_finder.fit(
    X_candidates.loc[:, compact_features],
    y=y_true,
    base_pred=oof_pred,
    sample_weight=sample_weight,
    splits=tvh_splits,
    original_model_features=base_feature_cols,
)

display(custom_split_finder.get_summary())
display(custom_split_finder.bootstrap_results_['split_role'].value_counts())


## Interpreting the output

Important V2 columns:

- `mean_oof_residual_r2`: average out-of-sample residual lift from a univariate residual model.
- `p05_oof_residual_r2` and `p95_oof_residual_r2`: bootstrap tail values for lift.
- `mean_residual_signal_rank` and `median_residual_signal_rank`: rank stability.
- `mean_effect_curve_spearman_stability`: average shape consistency across bootstraps.
- `null_beat_rate`: how often the real feature beats the 95th percentile shadow score.
- Null comparison plots show whether real feature lift beats shadow-feature lift.